In [ ]:
"""
MSTFDN — Cross-Session LOSO Training
=====================================
Leave-One-Session-Out (LOSO) strategy as described in paper Sec. 3.3.1:

  "One strategy was leave-one-session-out (LOSO) testing. Since each task
   consists of three sessions, the data from one session for each participant
   were used as the test set, while the remaining two sessions served as the
   training set."

3 folds per subject:
  Fold 1: train=[S2,S3]  test=[S1]
  Fold 2: train=[S1,S3]  test=[S2]
  Fold 3: train=[S1,S2]  test=[S3]

Metrics per subject: Accuracy, Recall, Precision, F1-score (average of 3 folds)
Final result: average across all subjects

Output files saved in OUTPUT_PATH/LOSO/:
  results_per_subject.csv   — per-subject per-fold metrics
  results_summary.csv       — per-subject average + all-subject average
  best_model_VP{id}_fold{f}.pt — best checkpoint per subject per fold
  training_log.txt          — timestamped log
"""

import os
import gc
import sys
import json
import time
import datetime
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR

warnings.filterwarnings('ignore')

# ── import model ──────────────────────────────────────────────────────────────
# فایل MSTFDN_model.py باید در همین پوشه باشد
sys.path.append(os.path.dirname(os.path.abspath(__file__)))
from MSTFDN_model_v2 import MSTFDN, MSTFDNConfig, MSTFDNLoss

# ===========================================================================
#  CONFIGURATION — تنظیم کن قبل از اجرا
# ===========================================================================
# آدرس پوشه‌ای که preprocessing خروجی‌ها را ذخیره کرده
# مثال Windows: r"E:\kntu\Thesis\codes\Results\MSTFDN_PREPROCESSED_NBACK"
# مثال Linux:   "/media/hassan-hossein/B660FEE360FEA8EF/Mahi/preprocessed"
PREPROCESSED_PATH = r"E:\kntu\Thesis\codes\Results\MSTFDN_PREPROCESSED_NBACK_V3"

# پوشه ذخیره نتایج
OUTPUT_PATH = r"E:\kntu\Thesis\codes\Results\MSTFDN_LOSO_V3_3CLASS"

# لیست همه subject ها
ALL_SUBJECTS = [f"VP{i:03d}" for i in range(1, 27)]

# اگر می‌خواهی فقط چند subject تست کنی:
# ALL_SUBJECTS = ["VP001", "VP002"]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ===========================================================================
#  MODEL CONFIG (N-back dataset — از مقاله Table A.1)
# ===========================================================================
def get_config() -> MSTFDNConfig:
    cfg = MSTFDNConfig()
    cfg.n_eeg_ch    = 28
    cfg.n_fnirs_ch  = 36
    cfg.T_e         = 2000    # 10s x 200Hz
    cfg.T_f         = 100     # 10s x 10Hz -- single window, matches paper exactly
                              # (Sec 3.2: fNIRS sample = 3x36x100, no 21-segment stack)
    cfg.n_bands     = 5
    cfg.n_hemo      = 3       # HbO, HbR, HbT -- all supplied as model input
    cfg.n_class     = 3       # 0-back, 2-back, 3-back ONLY -- 3 cognitive
                              # workload levels. REST (label=3, from paper's
                              # 4-class protocol) is EXCLUDED here, filtered
                              # out at the dataset level (see
                              # MSTFDNSubjectDataset below) -- preprocessing
                              # v3 still extracts all 4 classes on disk, we
                              # just drop the rest-class windows at load time.
    cfg.He          = 8       # T_e/He = 2000/8 = 250 (exact)
    cfg.Hf          = 5       # T_f/Hf = 100/5 = 20 (exact)
    cfg.Hef         = 5
    cfg.Ne          = 1
    cfg.Nf          = 1
    cfg.Nef         = 1
    cfg.KSS         = 4
    cfg.ffn_dim     = 256
    cfg.dropout     = 0.4
    cfg.lambda_nw   = 0.02
    cfg.lambda_mhs  = 0.05
    cfg.lr          = 3e-4
    cfg.weight_decay = 1e-3
    cfg.batch_size  = 16
    cfg.max_epochs  = 1000
    cfg.warmup_epochs = 10
    cfg.eta_min_ratio = 0.05
    cfg.patience    = 50
    return cfg

# ===========================================================================
#  DATASET
# ===========================================================================
class MSTFDNSubjectDataset(Dataset):
    """
    یک subject را load می‌کند -- خروجی MSTFDN_preprocessing_v3.py.

    EEG:   (N, 5, 28, 2000) float16  -- 5 باند فرکانسی واقعی، channel-space
           (بدون 16x16 grid -- مقاله اصلاً spatial mapping ندارد)
    fNIRS: (N, 3, 36, 100)  float32  -- HbO/HbR/HbT، یک window 10 ثانیه‌ای
           (بدون 21-segment stack -- طبق Sec 3.2 مقاله: X_fnirs در R^(3xCHfxTf))
    labels روی دیسک: 0=0-back, 1=2-back, 2=3-back, 3=rest  (4 کلاس، خروجی preprocessing)
    session_ids: (N,)  values 1/2/3

    exclude_rest=True (پیش‌فرض): نمونه‌های rest (label==3) در همین‌جا حذف
    می‌شوند و فقط ۳ سطح بار شناختی (0/2/3-back) باقی می‌ماند -- بدون نیاز
    به اجرای دوباره preprocessing.

    حافظه: eeg به‌صورت float16 نگه داشته می‌شود، تبدیل به float32 فقط در
    __getitem__ برای یک window انجام می‌شود -- نه برای کل subject.
    """
    REST_LABEL_ON_DISK = 3   # must match REST_LABEL in MSTFDN_preprocessing_v3.py

    def __init__(self, data_dir: str, subject_id: str, exclude_rest: bool = True):
        sub_dir = os.path.join(data_dir, subject_id)
        eeg    = np.load(os.path.join(sub_dir, f"{subject_id}_eeg_windows.npy"))   # (N,5,28,2000) float16
        fnirs  = np.load(os.path.join(sub_dir, f"{subject_id}_fnirs_windows.npy")) # (N,3,36,100)  float32
        labels = np.load(os.path.join(sub_dir, f"{subject_id}_labels.npy"))
        sess   = np.load(os.path.join(sub_dir, f"{subject_id}_session_ids.npy"))

        # Sanity check: reject old-format files (16x16 grid or 21-segment
        # stack) loudly instead of silently reshaping garbage.
        if eeg.ndim != 4 or eeg.shape[1] != 5:
            raise ValueError(
                f"{subject_id}: expected eeg_windows shape (N,5,CH,T_e) from "
                f"MSTFDN_preprocessing_v3.py, got {eeg.shape}. Old preprocessing "
                f"versions (16x16 grid or broadcast-copy bands) are not compatible."
            )
        if fnirs.ndim != 4 or fnirs.shape[1] != 3:
            raise ValueError(
                f"{subject_id}: expected fnirs_windows shape (N,3,CH,T_f) from "
                f"MSTFDN_preprocessing_v3.py, got {fnirs.shape}. Old preprocessing "
                f"versions (21-segment stack, 16x16 grid) are not compatible."
            )

        # Drop REST windows -> 3-class problem (0-back, 2-back, 3-back).
        # Labels stay 0/1/2 unchanged since rest was already the LAST label
        # index (3); no relabeling needed for the remaining 3 classes.
        if exclude_rest:
            keep = labels != self.REST_LABEL_ON_DISK
            eeg    = eeg[keep]
            fnirs  = fnirs[keep]
            labels = labels[keep]
            sess   = sess[keep]

        # Keep EEG as float16 in host RAM; cast to float32 per-window in
        # __getitem__ instead of for the whole subject.
        self.eeg         = torch.from_numpy(eeg).half()
        self.fnirs       = torch.from_numpy(fnirs).float()
        self.labels      = torch.from_numpy(labels).long()
        self.session_ids = sess   # numpy array, kept for splitting

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Cast EEG to float32 here (single window, cheap) -- the model
        # expects float32 input; keeping the full dataset in float16 in
        # RAM is what actually saves memory.
        return self.eeg[idx].float(), self.fnirs[idx], self.labels[idx]


# ===========================================================================
#  LOGGING
# ===========================================================================
LOG_FILE = None

def log(msg: str):
    ts   = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line, flush=True)
    if LOG_FILE:
        with open(LOG_FILE, "a", encoding="utf-8") as f:
            f.write(line + "\n")


# ===========================================================================
#  TRAINING UTILITIES
# ===========================================================================
def warmup_lr(optimizer, epoch: int, cfg: MSTFDNConfig):
    if epoch < cfg.warmup_epochs:
        scale = (epoch + 1) / cfg.warmup_epochs
        for pg in optimizer.param_groups:
            pg['lr'] = cfg.lr * scale


def train_one_epoch(model, loader, optimizer, loss_fn, device) -> dict:
    model.train()
    total_loss = ce_sum = l2_sum = mhs_sum = 0.0
    correct = n = 0

    for eeg, fnirs, labels in loader:
        eeg    = eeg.to(device)
        fnirs  = fnirs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits, he, hf, hef = model(eeg, fnirs)
        loss, ce, l2, mhs   = loss_fn(logits, labels, he, hf, hef)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        B = labels.size(0)
        total_loss += loss.item() * B
        ce_sum     += ce.item()   * B
        l2_sum     += l2.item()   * B
        mhs_sum    += mhs.item()  * B
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += B

    return {'loss': total_loss/n, 'ce': ce_sum/n,
            'l2': l2_sum/n, 'mhs': mhs_sum/n, 'acc': correct/n}


@torch.no_grad()
def evaluate(model, loader, loss_fn, device) -> dict:
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for eeg, fnirs, labels in loader:
        eeg    = eeg.to(device)
        fnirs  = fnirs.to(device)
        labels = labels.to(device)

        logits, he, hf, hef = model(eeg, fnirs)
        loss, *_ = loss_fn(logits, labels, he, hf, hef)

        total_loss += loss.item() * labels.size(0)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    n      = len(all_labels)
    acc    = accuracy_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    prec   = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    f1     = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    return {
        'loss': total_loss / n,
        'acc':  acc,
        'recall': recall,
        'precision': prec,
        'f1': f1,
        'preds':  all_preds,
        'labels': all_labels,
    }


def train_fold(model, cfg, train_loader, val_loader,
               checkpoint_path: str, device: str) -> dict:
    """
    Train one LOSO fold. Returns best val metrics.
    """
    loss_fn   = MSTFDNLoss(cfg, model)
    optimizer = torch.optim.Adam(model.parameters(),
                                  lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=cfg.max_epochs - cfg.warmup_epochs,
        eta_min=cfg.lr * cfg.eta_min_ratio)

    best_val_loss = float('inf')
    best_metrics  = {}
    patience_cnt  = 0

    for epoch in range(cfg.max_epochs):
        warmup_lr(optimizer, epoch, cfg)
        train_metrics = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        val_metrics   = evaluate(model, val_loader, loss_fn, device)

        if epoch >= cfg.warmup_epochs:
            scheduler.step()

        if epoch % 20 == 0:
            lr_now = optimizer.param_groups[0]['lr']
            log(f"    Ep{epoch:4d} | LR={lr_now:.2e} | "
                f"Train Loss={train_metrics['loss']:.4f} Acc={train_metrics['acc']:.4f} | "
                f"Val Loss={val_metrics['loss']:.4f} Acc={val_metrics['acc']:.4f} "
                f"F1={val_metrics['f1']:.4f}")

        # early stopping
        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            best_metrics  = val_metrics.copy()
            patience_cnt  = 0
            torch.save({
                'epoch':       epoch,
                'model_state': model.state_dict(),
                'val_metrics': val_metrics,
            }, checkpoint_path)
        else:
            patience_cnt += 1
            if patience_cnt >= cfg.patience:
                log(f"    Early stop at epoch {epoch}")
                break

    return best_metrics


# ===========================================================================
#  LOSO TRAINING FOR ONE SUBJECT
# ===========================================================================
def run_loso_subject(subject_id: str, cfg: MSTFDNConfig,
                     out_dir: str, device: str) -> list:
    """
    Run 3-fold LOSO for one subject.
    Returns list of 3 dicts (one per fold) with metrics.
    """
    log(f"\n{'='*55}")
    log(f"Subject: {subject_id}  |  LOSO (3 folds)")
    log(f"{'='*55}")

    # Load dataset
    dataset = MSTFDNSubjectDataset(
        data_dir=PREPROCESSED_PATH,
        subject_id=subject_id,
    )

    session_ids = dataset.session_ids   # numpy (N,) values 1/2/3
    sessions    = [1, 2, 3]
    fold_results = []

    for fold_idx, test_session in enumerate(sessions, 1):
        log(f"\n  Fold {fold_idx}/3 — Test session: S{test_session} | "
            f"Train sessions: {[s for s in sessions if s != test_session]}")

        # Split indices by session
        test_idx  = np.where(session_ids == test_session)[0]
        train_idx = np.where(session_ids != test_session)[0]

        log(f"  Train windows: {len(train_idx)} | Test windows: {len(test_idx)}")

        if len(train_idx) == 0 or len(test_idx) == 0:
            log(f"  WARNING: empty split for fold {fold_idx}, skipping")
            continue

        train_ds = Subset(dataset, train_idx)
        test_ds  = Subset(dataset, test_idx)

        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                                  shuffle=True, num_workers=0, pin_memory=True)
        test_loader  = DataLoader(test_ds, batch_size=cfg.batch_size,
                                  shuffle=False, num_workers=0, pin_memory=True)

        # Fresh model for each fold (paper: "trained from scratch")
        model    = MSTFDN(cfg).to(device)
        ckpt_path = os.path.join(out_dir, f"best_model_{subject_id}_fold{fold_idx}.pt")

        t0 = time.time()
        best_metrics = train_fold(model, cfg, train_loader, test_loader,
                                   ckpt_path, device)
        elapsed = (time.time() - t0) / 60

        fold_results.append({
            'subject':   subject_id,
            'fold':      fold_idx,
            'test_session': test_session,
            'train_sessions': str([s for s in sessions if s != test_session]),
            'n_train':   len(train_idx),
            'n_test':    len(test_idx),
            'accuracy':  round(best_metrics.get('acc', 0) * 100, 2),
            'recall':    round(best_metrics.get('recall', 0) * 100, 2),
            'precision': round(best_metrics.get('precision', 0) * 100, 2),
            'f1':        round(best_metrics.get('f1', 0) * 100, 2),
            'time_min':  round(elapsed, 2),
        })

        log(f"  Fold {fold_idx} done in {elapsed:.1f} min | "
            f"Acc={best_metrics.get('acc',0)*100:.2f}% "
            f"F1={best_metrics.get('f1',0)*100:.2f}%")

        # Free memory between folds
        del model, train_ds, test_ds, train_loader, test_loader
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()

    return fold_results


# ===========================================================================
#  MAIN — run all subjects
# ===========================================================================
def main():
    global LOG_FILE

    os.makedirs(OUTPUT_PATH, exist_ok=True)
    LOG_FILE = os.path.join(OUTPUT_PATH, "training_log.txt")

    log("=" * 55)
    log("MSTFDN — Cross-Session LOSO Training")
    log(f"Device : {DEVICE}")
    if DEVICE == 'cuda':
        log(f"GPU    : {torch.cuda.get_device_name(0)}")
        log(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    log(f"Subjects: {len(ALL_SUBJECTS)}")
    log(f"Preprocessed data: {PREPROCESSED_PATH}")
    log(f"Output: {OUTPUT_PATH}")
    log("=" * 55)

    cfg = get_config()
    all_fold_results = []

    total_t0 = time.time()

    for subject_id in ALL_SUBJECTS:
        sub_dir = os.path.join(PREPROCESSED_PATH, subject_id)
        if not os.path.exists(sub_dir):
            log(f"SKIPPED {subject_id}: data not found")
            continue

        try:
            fold_results = run_loso_subject(subject_id, cfg, OUTPUT_PATH, DEVICE)
            all_fold_results.extend(fold_results)

            # per-subject average across 3 folds
            if fold_results:
                avg_acc  = np.mean([r['accuracy']  for r in fold_results])
                avg_rec  = np.mean([r['recall']    for r in fold_results])
                avg_prec = np.mean([r['precision'] for r in fold_results])
                avg_f1   = np.mean([r['f1']        for r in fold_results])
                log(f"\n  {subject_id} LOSO average → "
                    f"Acc={avg_acc:.2f}% Recall={avg_rec:.2f}% "
                    f"Precision={avg_prec:.2f}% F1={avg_f1:.2f}%")

        except Exception as e:
            log(f"ERROR {subject_id}: {e}")
            import traceback; traceback.print_exc()

        finally:
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()

    # ── Save results ──────────────────────────────────────────────────────
    if not all_fold_results:
        log("No results to save.")
        return

    df_folds = pd.DataFrame(all_fold_results)
    df_folds.to_csv(os.path.join(OUTPUT_PATH, "results_per_subject.csv"), index=False)
    log(f"\nSaved: results_per_subject.csv")

    # Per-subject summary (mean ± std across 3 folds)
    summary_rows = []
    for sub in df_folds['subject'].unique():
        sub_df = df_folds[df_folds['subject'] == sub]
        summary_rows.append({
            'subject':       sub,
            'acc_mean':      round(sub_df['accuracy'].mean(), 2),
            'acc_std':       round(sub_df['accuracy'].std(), 2),
            'recall_mean':   round(sub_df['recall'].mean(), 2),
            'recall_std':    round(sub_df['recall'].std(), 2),
            'precision_mean':round(sub_df['precision'].mean(), 2),
            'precision_std': round(sub_df['precision'].std(), 2),
            'f1_mean':       round(sub_df['f1'].mean(), 2),
            'f1_std':        round(sub_df['f1'].std(), 2),
        })

    df_summary = pd.DataFrame(summary_rows)

    # All-subject average (final reported number — paper Table 2)
    overall = {
        'subject':       'ALL_SUBJECTS_AVG',
        'acc_mean':      round(df_summary['acc_mean'].mean(), 2),
        'acc_std':       round(df_summary['acc_mean'].std(), 2),
        'recall_mean':   round(df_summary['recall_mean'].mean(), 2),
        'recall_std':    round(df_summary['recall_mean'].std(), 2),
        'precision_mean':round(df_summary['precision_mean'].mean(), 2),
        'precision_std': round(df_summary['precision_mean'].std(), 2),
        'f1_mean':       round(df_summary['f1_mean'].mean(), 2),
        'f1_std':        round(df_summary['f1_mean'].std(), 2),
    }
    df_summary = pd.concat([df_summary, pd.DataFrame([overall])], ignore_index=True)
    df_summary.to_csv(os.path.join(OUTPUT_PATH, "results_summary.csv"), index=False)
    log(f"Saved: results_summary.csv")

    total_elapsed = (time.time() - total_t0) / 60
    log(f"\n{'='*55}")
    log(f"LOSO FINISHED in {total_elapsed:.1f} min")
    log(f"Final — Acc: {overall['acc_mean']} ± {overall['acc_std']}%  "
        f"F1: {overall['f1_mean']} ± {overall['f1_std']}%")
    log(f"{'='*55}")

    print("\n=== FINAL RESULTS ===")
    print(df_summary.to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
"""
mstfdn_diagnostics.py
================================
Comprehensive diagnostic visualization for MSTFDN (MSTFDN_model_v2.py).

REWRITTEN from the TSFNet_With_fNIRS_2D diagnostics script to match the
ACTUAL MSTFDN architecture:

  - EEG branch:    E-TFEM -> E-SFEM -> E-STFEM -> f_eeg_st (KSS*CH_e, T_f)
  - fNIRS branch:  F-TFEM -> F-SFEM -> F-STFEM -> f_fnirs_st (KSS*CH_f, T_f)
  - Fusion branch: concat -> EFF-SFEM -> f_fusion_s (KSS*(CH_e+CH_f), T_f)
  - Classifier:    cls_conv1 -> ReLU -> cls_conv2 -> ReLU -> GAP -> Softmax
                   ONE classifier only (unlike TSFNet's three separate
                   fc_eeg/fc_fnirs/fc_fusion heads) -- so there is only a
                   SINGLE "Final" confusion matrix, not four per-branch ones.
  - forward() returns (logits, heads_e, heads_f, heads_ef) -- a 4-tuple,
    not TSFNet's 8-tuple.
  - 4 classes for N-back: 0-back, 2-back, 3-back, rest (paper Sec 3.2).

KEY DIFFERENCES vs the original TSFNet diagnostics script:
  1. No per-branch classifier heads exist -> only ONE confusion matrix
     ("Final"), and per-class accuracy is computed from the single final
     classifier, not compared across 4 separate branch heads.
  2. Branch FEATURES still exist (f_eeg_st, f_fnirs_st, f_fusion_s) and are
     extracted via forward hooks on e_stfem / f_stfem / eff_sfem, then
     GAP-pooled over the time dimension T_f to get a fixed-size vector
     per branch for PCA/t-SNE (these modules output (B, C, T_f), not a
     ready-made (B, 64) vector like TSFNet's Linear layers).
  3. No BiLSTM/cudnn RNN in MSTFDN -> gradient saliency does NOT need the
     model.train()-with-frozen-BN workaround; plain model.eval() works.
  4. EEG input shape is (B, 5, 28, 2000) [band, channel, time], fNIRS is
     (B, 3, 36, 100) [hemo, channel, time] -- saliency plots are adapted
     to these shapes (band-averaged EEG saliency over the 10s window).
  5. CLASS_NAMES updated to 4 classes.

HOW TO USE
----------
    from mstfdn_diagnostics import run_diagnostics

    model = MSTFDN(cfg)
    model.load_state_dict(torch.load(checkpoint_path)['model_state'])

    run_diagnostics(
        model        = model,
        train_loader = train_loader,
        test_loader  = test_loader,
        device       = device,
        subject_id   = "VP001",
        fold_id      = 1,
        output_dir   = os.path.join(OUTPUT_PATH, "diagnostics", "VP001_fold1"),
    )

This is a POST-HOC analysis tool -- run it AFTER training finishes (e.g.
right after loading a saved best-model checkpoint from
MSTFDN_train_LOSO_v2.py), not during the training loop itself. To run it
for several subjects for a report, loop over subjects and call
run_diagnostics() once per (subject, fold) with a fresh output_dir.
"""

import os
import re
import gc
import warnings
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (confusion_matrix, silhouette_score,
                              davies_bouldin_score, classification_report)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

CLASS_NAMES = ["0-back", "2-back", "3-back", "rest"]   # paper Sec 3.2, 4 classes
N_CLASS     = len(CLASS_NAMES)
COLORS      = ["#2a78d6", "#e24b4a", "#1baf7a", "#9b59b6"]


# ---------------------------------------------------------------------------
# 1. Feature extraction via forward hooks on the 3 branch output modules
#    (e_stfem, f_stfem, eff_sfem), GAP-pooled over T_f to a fixed vector.
# ---------------------------------------------------------------------------

def _extract_features(model, loader, device):
    """
    One forward pass over `loader`, collecting:
      feat_eeg    : (N, KSS*CH_e)          -- GAP-pooled E-STFEM output
      feat_fnirs  : (N, KSS*CH_f)          -- GAP-pooled F-STFEM output
      feat_fusion : (N, KSS*(CH_e+CH_f))   -- GAP-pooled EFF-SFEM output
      feat_combined : concat of the three
      logits      : (N, N_CLASS)           -- final classifier output
      labels      : (N,)
    """
    model.eval()
    feat_store = {"eeg": [], "fnirs": [], "fusion": []}

    def _hook_factory(key):
        def _hook(module, inp, out):
            # out shape: (B, C, T_f) for e_stfem/f_stfem/eff_sfem -> GAP over T_f
            pooled = out.mean(dim=-1).detach().cpu().numpy()   # (B, C)
            feat_store[key].append(pooled)
        return _hook

    h1 = model.e_stfem.register_forward_hook(_hook_factory("eeg"))
    h2 = model.f_stfem.register_forward_hook(_hook_factory("fnirs"))
    # eff_sfem returns (x, head_outputs) as a tuple -- hook sees the whole
    # tuple as `out`, so index [0] for the feature tensor
    def _fusion_hook(module, inp, out):
        pooled = out[0].mean(dim=-1).detach().cpu().numpy()
        feat_store["fusion"].append(pooled)
    h3 = model.eff_sfem.register_forward_hook(_fusion_hook)

    all_logits, all_labels = [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in loader:
            logits, _, _, _ = model(eeg_b.to(device), nirs_b.to(device))
            all_logits.append(logits.cpu().numpy())
            all_labels.append(lbl_b.numpy())

    h1.remove(); h2.remove(); h3.remove()

    feat_eeg    = np.concatenate(feat_store["eeg"],    axis=0)
    feat_fnirs  = np.concatenate(feat_store["fnirs"],  axis=0)
    feat_fusion = np.concatenate(feat_store["fusion"], axis=0)
    logits      = np.concatenate(all_logits, axis=0)
    labels      = np.concatenate(all_labels, axis=0)
    feat_combined = np.concatenate([feat_eeg, feat_fnirs, feat_fusion], axis=1)

    del feat_store, all_logits, all_labels
    gc.collect()

    return {
        "feat_eeg": feat_eeg, "feat_fnirs": feat_fnirs, "feat_fusion": feat_fusion,
        "feat_combined": feat_combined, "logits": logits, "labels": labels,
    }


# ---------------------------------------------------------------------------
# 2. Confusion matrix -- ONLY ONE (MSTFDN has a single classifier, unlike
#    TSFNet's separate per-branch heads)
# ---------------------------------------------------------------------------

def _plot_confusion(logits, labels, tag, output_dir):
    preds = np.argmax(logits, axis=1)
    cm    = confusion_matrix(labels, preds, labels=list(range(N_CLASS)))
    cmn   = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100

    fig, ax = plt.subplots(figsize=(6, 5.5))
    im = ax.imshow(cmn, vmin=0, vmax=100, cmap="Blues")
    ax.set_xticks(range(N_CLASS)); ax.set_yticks(range(N_CLASS))
    ax.set_xticklabels(CLASS_NAMES, fontsize=9, rotation=30)
    ax.set_yticklabels(CLASS_NAMES, fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix (acc={cm.diagonal().sum()/cm.sum()*100:.1f}%)", fontsize=10)
    for i in range(N_CLASS):
        for j in range(N_CLASS):
            ax.text(j, i, f"{cmn[i,j]:.1f}%\n({cm[i,j]})",
                    ha="center", va="center",
                    color="white" if cmn[i,j] > 55 else "black", fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(f"{tag} -- Final Classifier", fontsize=12, y=1.02)
    plt.tight_layout()
    path = os.path.join(output_dir, f"{tag}_01_confusion.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  [saved] {path}")


# ---------------------------------------------------------------------------
# 3. Per-class accuracy (single classifier -- no per-branch comparison
#    possible since MSTFDN has no separate branch heads)
# ---------------------------------------------------------------------------

def _plot_per_class_acc(logits, labels, tag, output_dir):
    preds = np.argmax(logits, axis=1)
    accs  = [(preds[labels == c] == c).mean() * 100 if (labels == c).sum() > 0 else 0
             for c in range(N_CLASS)]

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(CLASS_NAMES, accs, color=COLORS, alpha=0.85, edgecolor="white")
    for bar, v in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.5,
                f"{v:.1f}", ha="center", fontsize=9)
    ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0, 105)
    ax.axhline(100.0 / N_CLASS, ls="--", color="gray", lw=0.8,
               label=f"Chance ({100.0/N_CLASS:.1f}%)")
    ax.legend(fontsize=9)
    ax.set_title(f"{tag} -- Per-class Accuracy")
    plt.tight_layout()
    path = os.path.join(output_dir, f"{tag}_02_per_class_acc.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  [saved] {path}")


# ---------------------------------------------------------------------------
# 4. PCA -- per branch feature space (eeg_st, fnirs_st, fusion)
# ---------------------------------------------------------------------------

def _plot_pca(train_data, test_data, tag, output_dir):
    for feat_key, title in [("feat_eeg",      "EEG branch (E-STFEM, GAP-pooled)"),
                              ("feat_fnirs",    "fNIRS branch (F-STFEM, GAP-pooled)"),
                              ("feat_fusion",   "Fusion branch (EFF-SFEM, GAP-pooled)"),
                              ("feat_combined", "All branches concatenated")]:
        X_tr, y_tr = train_data[feat_key], train_data["labels"]
        X_te, y_te = test_data[feat_key],  test_data["labels"]

        scaler = StandardScaler().fit(X_tr)
        pca    = PCA(n_components=2, random_state=42).fit(scaler.transform(X_tr))
        Z_tr   = pca.transform(scaler.transform(X_tr))
        Z_te   = pca.transform(scaler.transform(X_te))

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for ax, (Z, y, split) in zip(axes, [(Z_tr, y_tr, "Train"), (Z_te, y_te, "Test")]):
            for c, (cname, col) in enumerate(zip(CLASS_NAMES, COLORS)):
                mask = y == c
                ax.scatter(Z[mask,0], Z[mask,1], s=8, alpha=0.5, color=col, label=cname)
            ax.set_title(f"PCA -- {split} | {title}")
            ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
            ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
            ax.legend(markerscale=3, fontsize=8)

        plt.suptitle(f"{tag} -- PCA Feature Space", fontsize=12)
        plt.tight_layout()
        fkey = feat_key.replace("feat_", "")
        path = os.path.join(output_dir, f"{tag}_03_pca_{fkey}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
        print(f"  [saved] {path}")


# ---------------------------------------------------------------------------
# 5. t-SNE + misclassified overlay
# ---------------------------------------------------------------------------

def _plot_tsne(test_data, tag, output_dir, perplexity=30):
    for feat_key, title in [("feat_eeg",      "EEG branch (E-STFEM)"),
                              ("feat_fnirs",    "fNIRS branch (F-STFEM)"),
                              ("feat_fusion",   "Fusion branch (EFF-SFEM)")]:
        X    = test_data[feat_key]
        y    = test_data["labels"]
        pred = np.argmax(test_data["logits"], axis=1)

        X_s = StandardScaler().fit_transform(X)
        n_max = min(3000, len(X_s))
        idx   = np.random.choice(len(X_s), n_max, replace=False)
        X_s, y_s, pred_s = X_s[idx], y[idx], pred[idx]

        print(f"    t-SNE {feat_key} ({n_max} samples)...", end="", flush=True)
        Z = TSNE(n_components=2, perplexity=min(perplexity, max(5, n_max // 4)),
                 random_state=42, max_iter=1000).fit_transform(X_s)
        print(" done")

        wrong = pred_s != y_s
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        ax = axes[0]
        for c, (cname, col) in enumerate(zip(CLASS_NAMES, COLORS)):
            mask = y_s == c
            ax.scatter(Z[mask,0], Z[mask,1], s=7, alpha=0.45, color=col, label=cname)
        ax.set_title("Colour = true class"); ax.legend(markerscale=3, fontsize=9)

        ax = axes[1]
        ax.scatter(Z[~wrong,0], Z[~wrong,1], s=6, alpha=0.35, color="#1baf7a", label="Correct")
        ax.scatter(Z[wrong,0], Z[wrong,1], s=10, alpha=0.7, color="#e24b4a",
                   label="Misclassified", marker="x")
        ax.set_title(f"Correct vs Misclassified (error {wrong.mean()*100:.1f}%)")
        ax.legend(markerscale=3, fontsize=9)

        fig.suptitle(f"{tag} -- t-SNE -- {title}", fontsize=12)
        plt.tight_layout()
        fkey = feat_key.replace("feat_", "")
        path = os.path.join(output_dir, f"{tag}_04_tsne_{fkey}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
        print(f"  [saved] {path}")


# ---------------------------------------------------------------------------
# 6. Train vs Test distribution shift
# ---------------------------------------------------------------------------

def _plot_distribution_shift(train_data, test_data, tag, output_dir):
    for feat_key, title in [("feat_eeg",    "EEG branch (E-STFEM)"),
                              ("feat_fnirs",  "fNIRS branch (F-STFEM)")]:
        X_tr = train_data[feat_key]; X_te = test_data[feat_key]
        mu_tr, mu_te   = X_tr.mean(0), X_te.mean(0)
        std_tr, std_te = X_tr.std(0),  X_te.std(0)
        dims = np.arange(X_tr.shape[1])

        fig, axes = plt.subplots(2, 1, figsize=(14, 8))
        ax = axes[0]
        ax.plot(dims, mu_tr, color="#2a78d6", lw=1.0, label="Train mean")
        ax.plot(dims, mu_te, color="#e24b4a", lw=1.0, label="Test mean")
        ax.fill_between(dims, mu_tr-std_tr, mu_tr+std_tr, color="#2a78d6", alpha=0.15)
        ax.fill_between(dims, mu_te-std_te, mu_te+std_te, color="#e24b4a", alpha=0.15)
        ax.set_xlabel("Feature dimension (KSS * channels)"); ax.set_ylabel("Activation")
        ax.set_title(f"Mean +/- Std -- {title}"); ax.legend()

        ax = axes[1]
        shift = np.abs(mu_tr - mu_te)
        ax.bar(dims, shift, color="#ef9f27", width=0.8)
        ax.set_xlabel("Feature dimension")
        ax.set_ylabel("|Train mean - Test mean|")
        ax.set_title("Distribution shift per dimension (high = session-to-session shift)")
        top5 = np.argsort(shift)[-5:]
        for d in top5:
            ax.axvline(d, color="red", lw=0.8, ls="--")

        plt.suptitle(f"{tag} -- Distribution Shift", fontsize=12)
        plt.tight_layout()
        fkey = feat_key.replace("feat_", "")
        path = os.path.join(output_dir, f"{tag}_05_dist_shift_{fkey}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
        print(f"  [saved] {path}")


# ---------------------------------------------------------------------------
# 7. Class centroid distance matrix (4x4 now)
# ---------------------------------------------------------------------------

def _plot_centroid_distance(train_data, test_data, tag, output_dir):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax, (data, split) in zip(axes, [(train_data, "Train"), (test_data, "Test")]):
        X = data["feat_combined"]; y = data["labels"]
        centroids = np.array([
            X[y == c].mean(0) if (y == c).sum() > 0 else np.zeros(X.shape[1])
            for c in range(N_CLASS)
        ])
        dist = np.array([[np.linalg.norm(centroids[i]-centroids[j]) for j in range(N_CLASS)]
                         for i in range(N_CLASS)])
        im = ax.imshow(dist, cmap="YlOrRd")
        ax.set_xticks(range(N_CLASS)); ax.set_yticks(range(N_CLASS))
        ax.set_xticklabels(CLASS_NAMES, rotation=30); ax.set_yticklabels(CLASS_NAMES)
        for i in range(N_CLASS):
            for j in range(N_CLASS):
                ax.text(j, i, f"{dist[i,j]:.2f}", ha="center", va="center",
                        fontsize=9, fontweight="bold")
        plt.colorbar(im, ax=ax)
        ax.set_title(f"{split} -- Centroid Distance")

    plt.suptitle(f"{tag} -- Class Centroid Distance (combined features)", fontsize=12)
    plt.tight_layout()
    path = os.path.join(output_dir, f"{tag}_06_centroid_dist.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  [saved] {path}")


# ---------------------------------------------------------------------------
# 8. Confidence & entropy
# ---------------------------------------------------------------------------

def _plot_confidence(test_data, tag, output_dir):
    logits = test_data["logits"]; labels = test_data["labels"]
    probs   = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds   = np.argmax(logits, axis=1)
    correct = preds == labels

    max_prob = probs.max(axis=1)
    entropy  = -(probs * np.log(probs + 1e-9)).sum(axis=1)
    max_ent  = np.log(N_CLASS)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(max_prob[correct],  bins=40, alpha=0.7, color="#1baf7a",
            label=f"Correct ({correct.sum()})", density=True)
    ax.hist(max_prob[~correct], bins=40, alpha=0.7, color="#e24b4a",
            label=f"Wrong ({(~correct).sum()})", density=True)
    ax.set_xlabel("Max softmax probability"); ax.set_ylabel("Density")
    ax.set_title("Confidence Distribution"); ax.legend()
    ax.axvline(1.0 / N_CLASS, color="gray", ls="--", lw=0.8)

    ax = axes[1]
    ax.hist(entropy[correct]/max_ent,  bins=40, alpha=0.7, color="#1baf7a",
            label="Correct", density=True)
    ax.hist(entropy[~correct]/max_ent, bins=40, alpha=0.7, color="#e24b4a",
            label="Wrong", density=True)
    ax.set_xlabel("Normalised entropy (0=certain, 1=uniform)")
    ax.set_ylabel("Density")
    ax.set_title("Prediction Uncertainty (Entropy)"); ax.legend()

    plt.suptitle(f"{tag} -- Confidence & Uncertainty", fontsize=12)
    plt.tight_layout()
    path = os.path.join(output_dir, f"{tag}_07_confidence.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  [saved] {path}")


# ---------------------------------------------------------------------------
# 9. Cluster metrics
# ---------------------------------------------------------------------------

def _compute_cluster_metrics(data):
    results = {}
    for feat_key in ["feat_eeg", "feat_fnirs", "feat_fusion", "feat_combined"]:
        X = data[feat_key]; y = data["labels"]
        n_classes_present = len(np.unique(y))
        if n_classes_present < 2:
            results[feat_key] = {"silhouette": float("nan"), "davies_bouldin": float("nan")}
            continue
        n   = min(2000, len(X))
        idx = np.random.choice(len(X), n, replace=False)
        Xs, ys = StandardScaler().fit_transform(X[idx]), y[idx]
        results[feat_key] = {
            "silhouette":     silhouette_score(Xs, ys, metric="euclidean"),
            "davies_bouldin": davies_bouldin_score(Xs, ys),
        }
    return results


# ---------------------------------------------------------------------------
# 10. Input gradient saliency (EEG) -- NO cudnn RNN issue in MSTFDN
#     (no LSTM anywhere), so plain model.eval() works for backward().
# ---------------------------------------------------------------------------

def _plot_saliency(model, test_data_loader, device, tag, output_dir, n_samples=200):
    model.eval()

    _grads = {c: [] for c in range(N_CLASS)}
    collected = 0

    for eeg_b, nirs_b, lbl_b in test_data_loader:
        if collected >= n_samples:
            break
        eeg_b  = eeg_b.to(device).float()
        nirs_b = nirs_b.to(device).float()
        lbl_np = lbl_b.numpy()
        eeg_b.requires_grad_(True)

        logits, _, _, _ = model(eeg_b, nirs_b)   # (B, N_CLASS)

        for c in range(N_CLASS):
            mask = (lbl_np == c)
            if mask.sum() == 0:
                continue
            if eeg_b.grad is not None:
                eeg_b.grad.zero_()
            model.zero_grad()
            score = logits[mask, c].sum()
            score.backward(retain_graph=(c < N_CLASS - 1))
            if eeg_b.grad is not None:
                grad = eeg_b.grad.detach().cpu().abs()   # (B, 5, 28, 2000)
                _grads[c].append(grad[mask].mean(0))       # (5, 28, 2000)

        eeg_b = eeg_b.detach()
        collected += len(lbl_np)

    if not any(_grads[c] for c in range(N_CLASS)):
        print("  [saliency] no gradients collected -- skipped.")
        return

    fig, axes = plt.subplots(N_CLASS, 1, figsize=(14, 3 * N_CLASS), sharex=True)
    for c, ax in enumerate(axes):
        if not _grads[c]:
            ax.set_ylabel(f"|Gradient|\n{CLASS_NAMES[c]}", fontsize=9)
            continue
        g = torch.stack(_grads[c]).mean(0)          # (5, 28, 2000) -- avg over collected samples
        g_time = g.mean(dim=(0, 1)).numpy()          # avg over band & channel -> (2000,)
        t = np.linspace(0, 10, len(g_time))          # 10s EEG window
        ax.plot(t, g_time, color=COLORS[c % len(COLORS)], lw=1.0)
        ax.fill_between(t, 0, g_time, color=COLORS[c % len(COLORS)], alpha=0.3)
        ax.set_ylabel(f"|Gradient|\n{CLASS_NAMES[c]}", fontsize=9)
        ax.set_xlim(0, 10)

    axes[-1].set_xlabel("Time within window (seconds)")
    plt.suptitle(f"{tag} -- Input Gradient Saliency (EEG, band+channel avg)\n"
                 "(which time points drive predictions)", fontsize=12)
    plt.tight_layout()
    path = os.path.join(output_dir, f"{tag}_08_saliency.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  [saved] {path}")


# ---------------------------------------------------------------------------
# 11. Text report
# ---------------------------------------------------------------------------

def _write_report(test_data, cluster_metrics, tag, output_dir):
    logits = test_data["logits"]; labels = test_data["labels"]
    preds  = np.argmax(logits, axis=1)

    lines = [
        f"DIAGNOSTIC REPORT -- {tag}", "=" * 60,
        "\n[Classification Report -- Final Classifier]",
        classification_report(labels, preds, target_names=CLASS_NAMES,
                              labels=list(range(N_CLASS)), zero_division=0),
        "\n[Cluster Separability Metrics]",
        f"{'Feature':20s}  {'Silhouette':>12}  {'Davies-Bouldin':>15}",
        "-" * 52,
    ]
    for k, v in cluster_metrics.items():
        tag2 = k.replace("feat_", "")
        lines.append(f"{tag2:20s}  {v['silhouette']:12.4f}  {v['davies_bouldin']:15.4f}")
    lines += [
        "\nNote: Silhouette in [-1,1] -- higher is better separable.",
        "      Davies-Bouldin >= 0    -- lower is better separable.",
        "\n[Model Architecture Notes -- MSTFDN]",
        "  EEG branch:   E-TFEM (4-scale dilated conv, groups=5 bands) ->",
        "                E-SFEM (Transformer, He heads, BN in FFN) ->",
        "                E-STFEM (depthwise stride=20 + pointwise KSS)",
        "  fNIRS branch: F-TFEM (2-scale dilated conv, groups=3 hemo) ->",
        "                F-SFEM (Transformer, Hf heads, LN in FFN) ->",
        "                F-STFEM (depthwise kernel=11 + pointwise KSS)",
        "  Fusion:       concat(eeg_st, fnirs_st) -> EFF-SFEM (Hef heads, LN) ->",
        "                Classifier: Conv1d->ReLU->Conv1d->ReLU->GAP->Softmax",
        "  Loss:         CrossEntropy + L2-norm(lambda_nw) + HSIC(lambda_mhs)",
        "  NOTE: MSTFDN has a SINGLE classifier (unlike TSFNet's 3 separate",
        "        branch heads), so only one confusion matrix/accuracy exists;",
        "        PCA/t-SNE are still shown per-branch using the intermediate",
        "        E-STFEM / F-STFEM / EFF-SFEM feature representations.",
    ]

    path = os.path.join(output_dir, f"{tag}_report.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"  [saved] {path}")
    print("\n".join(lines))


# ---------------------------------------------------------------------------
# MAIN ENTRY POINT
# ---------------------------------------------------------------------------

def run_diagnostics(model, train_loader, test_loader, device,
                    subject_id: str, fold_id: int, output_dir: str):
    """
    Parameters
    ----------
    model        : trained MSTFDN instance (already .to(device))
    train_loader : DataLoader for the training sessions of this fold
    test_loader  : DataLoader for the test session of this fold
    device       : 'cuda' or 'cpu'
    subject_id   : e.g. 'VP001'
    fold_id      : 1, 2, or 3 (LOSO fold number)
    output_dir   : folder where all PNGs and the report are saved

    Call this AFTER training (e.g. right after loading a saved checkpoint
    from MSTFDN_train_LOSO_v2.py), once per (subject, fold). To generate a
    report across several subjects, loop over subjects and call this once
    per subject with a distinct output_dir.
    """
    os.makedirs(output_dir, exist_ok=True)
    tag = f"{subject_id}_fold{fold_id}"
    print(f"\n{'='*60}")
    print(f"  MSTFDN DIAGNOSTICS -- {tag}")
    print(f"{'='*60}")

    print("\n[1/9] Extracting features -- train set...")
    train_data = _extract_features(model, train_loader, device)

    print("[2/9] Extracting features -- test set...")
    test_data  = _extract_features(model, test_loader, device)

    labels = test_data["labels"]
    logits = test_data["logits"]

    print("[3/9] Confusion matrix...")
    _plot_confusion(logits, labels, tag, output_dir)

    print("[4/9] Per-class accuracy...")
    _plot_per_class_acc(logits, labels, tag, output_dir)

    print("[5/9] PCA...")
    _plot_pca(train_data, test_data, tag, output_dir)

    print("[6/9] t-SNE (1-2 min)...")
    _plot_tsne(test_data, tag, output_dir)

    print("[7/9] Distribution shift...")
    _plot_distribution_shift(train_data, test_data, tag, output_dir)

    _plot_centroid_distance(train_data, test_data, tag, output_dir)
    _plot_confidence(test_data, tag, output_dir)

    print("[+] Cluster metrics...")
    cluster_metrics = _compute_cluster_metrics(test_data)

    print("[+] Gradient saliency (EEG)...")
    _plot_saliency(model, test_loader, device, tag, output_dir)

    print("[+] Writing report...")
    _write_report(test_data, cluster_metrics, tag, output_dir)

    del train_data, test_data, logits, labels
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"\n  All diagnostics saved -> {output_dir}\n")


# ---------------------------------------------------------------------------
# EXAMPLE: run diagnostics for multiple subjects (for a report)
# ---------------------------------------------------------------------------
#
#   import torch
#   from MSTFDN_model_v2 import MSTFDN
#   from MSTFDN_train_LOSO_v2 import get_config, MSTFDNSubjectDataset
#   from torch.utils.data import DataLoader, Subset
#   import numpy as np, os
#
#   PREPROCESSED_PATH = r"E:\...\MSTFDN_PREPROCESSED_NBACK_V3"
#   OUTPUT_PATH        = r"E:\...\MSTFDN_LOSO_V3"
#   device = 'cuda' if torch.cuda.is_available() else 'cpu'
#   cfg = get_config()
#
#   subjects_for_report = ["VP001", "VP002", "VP003"]
#   fold_id = 1   # test session S1, matches best_model_VPxxx_fold1.pt
#
#   for subject_id in subjects_for_report:
#       dataset = MSTFDNSubjectDataset(PREPROCESSED_PATH, subject_id)
#       sess = dataset.session_ids
#       test_idx  = np.where(sess == fold_id)[0]
#       train_idx = np.where(sess != fold_id)[0]
#       train_loader = DataLoader(Subset(dataset, train_idx), batch_size=16, shuffle=False)
#       test_loader  = DataLoader(Subset(dataset, test_idx),  batch_size=16, shuffle=False)
#
#       model = MSTFDN(cfg).to(device)
#       ckpt = torch.load(
#           os.path.join(OUTPUT_PATH, f"best_model_{subject_id}_fold{fold_id}.pt"),
#           map_location=device)
#       model.load_state_dict(ckpt['model_state'])
#
#       run_diagnostics(
#           model=model, train_loader=train_loader, test_loader=test_loader,
#           device=device, subject_id=subject_id, fold_id=fold_id,
#           output_dir=os.path.join(OUTPUT_PATH, "diagnostics", subject_id),
#       )


# ============================================================================
# VISUALIZATION DRIVER -- choose which subjects to visualize
# ============================================================================
# این بخش را برای انتخاب subject ها ویرایش کن.
#
#   SUBJECT_SELECTION_MODE = "random"  -> N_RANDOM_SUBJECTS تای رندوم انتخاب می‌شود
#   SUBJECT_SELECTION_MODE = "manual"  -> فقط لیست MANUAL_SUBJECT_LIST اجرا می‌شود
#
# نیاز به اجرای قبلی سلول training (Cell 1) دارد -- از آن این متغیرها/توابع
# استفاده می‌شود: get_config, MSTFDN, MSTFDNSubjectDataset, PREPROCESSED_PATH,
# OUTPUT_PATH, DEVICE, DataLoader, Subset, torch, np, os, gc
# ============================================================================

import random

SUBJECT_SELECTION_MODE = "random"     # "random" یا "manual"
N_RANDOM_SUBJECTS      = 5            # تعداد subject تصادفی (اگر mode="random")
MANUAL_SUBJECT_LIST    = ["VP001", "VP002", "VP003"]   # فقط اگر mode="manual"
RANDOM_SEED_VIZ        = 42           # برای تکرارپذیری انتخاب تصادفی
FOLD_ID_FOR_VIZ        = 1            # کدام fold (1, 2, یا 3) را visualize کنیم


def _get_available_subjects(output_path: str, fold_id: int) -> list:
    """subject هایی که checkpoint ذخیره‌شده برای این fold دارند."""
    pattern = re.compile(rf"best_model_(VP\d+)_fold{fold_id}\.pt")
    found = []
    if not os.path.exists(output_path):
        return found
    for fname in os.listdir(output_path):
        m = pattern.match(fname)
        if m:
            found.append(m.group(1))
    return sorted(found)


def run_visualization_for_selected_subjects():
    """
    نقطه ورود اصلی این سلول. subject ها را طبق تنظیمات بالا انتخاب کرده و
    برای هرکدام run_diagnostics() را با checkpoint ذخیره‌شده‌اش اجرا می‌کند.
    """
    available = _get_available_subjects(OUTPUT_PATH, FOLD_ID_FOR_VIZ)
    print(f"Subjects with a saved fold-{FOLD_ID_FOR_VIZ} checkpoint: "
          f"{len(available)} -> {available}")

    if not available:
        print(f"[ERROR] No checkpoints found in {OUTPUT_PATH} for fold {FOLD_ID_FOR_VIZ}. "
              f"Run the training cell first.")
        return

    if SUBJECT_SELECTION_MODE == "random":
        random.seed(RANDOM_SEED_VIZ)
        n = min(N_RANDOM_SUBJECTS, len(available))
        subjects_to_visualize = random.sample(available, n)
    elif SUBJECT_SELECTION_MODE == "manual":
        subjects_to_visualize = [s for s in MANUAL_SUBJECT_LIST if s in available]
        missing = [s for s in MANUAL_SUBJECT_LIST if s not in available]
        if missing:
            print(f"[WARNING] No checkpoint for: {missing} -- skipping those.")
    else:
        raise ValueError("SUBJECT_SELECTION_MODE must be 'random' or 'manual'")

    print(f"\nSelected subjects for visualization: {subjects_to_visualize}\n")

    cfg = get_config()

    for subject_id in subjects_to_visualize:
        print(f"\n{'='*60}")
        print(f"  Visualizing {subject_id}  (fold {FOLD_ID_FOR_VIZ})")
        print(f"{'='*60}")
        model = None
        try:
            dataset = MSTFDNSubjectDataset(PREPROCESSED_PATH, subject_id, exclude_rest=True)
            sess = dataset.session_ids
            test_idx  = np.where(sess == FOLD_ID_FOR_VIZ)[0]
            train_idx = np.where(sess != FOLD_ID_FOR_VIZ)[0]

            if len(test_idx) == 0 or len(train_idx) == 0:
                print(f"  [WARNING] Empty split for {subject_id}, skipping.")
                continue

            train_loader = DataLoader(Subset(dataset, train_idx),
                                      batch_size=cfg.batch_size, shuffle=False)
            test_loader  = DataLoader(Subset(dataset, test_idx),
                                      batch_size=cfg.batch_size, shuffle=False)

            model = MSTFDN(cfg).to(DEVICE)
            ckpt_path = os.path.join(OUTPUT_PATH, f"best_model_{subject_id}_fold{FOLD_ID_FOR_VIZ}.pt")
            ckpt = torch.load(ckpt_path, map_location=DEVICE)
            model.load_state_dict(ckpt['model_state'])

            run_diagnostics(
                model=model, train_loader=train_loader, test_loader=test_loader,
                device=DEVICE, subject_id=subject_id, fold_id=FOLD_ID_FOR_VIZ,
                output_dir=os.path.join(OUTPUT_PATH, "diagnostics", subject_id),
            )

        except Exception as e:
            print(f"  [WARNING] Diagnostics failed for {subject_id}: {e}")
            import traceback; traceback.print_exc()

        finally:
            if model is not None:
                del model
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()

    print(f"\n{'='*60}")
    print(f"  All {len(subjects_to_visualize)} selected subjects visualized.")
    print(f"  Output: {os.path.join(OUTPUT_PATH, 'diagnostics')}")
    print(f"{'='*60}")


# اجرای خودکار وقتی این سلول ران می‌شود
run_visualization_for_selected_subjects()
